# 🚗 Vehicle ReID Training - Complete Notebook

**Train a custom Vehicle Re-Identification model using TorchReID**

## Overview
- **Model**: OSNet-AIN (Omni-Scale Network with Attention)
- **Framework**: TorchReID
- **Dataset**: Custom vehicle dataset (CAR, LCV, TRUCK, BUS)
- **Output**: PyTorch model (.pth) + OpenVINO model (.xml)

## Steps
1. Setup Environment
2. Upload & Extract Dataset
3. Analyze Dataset
4. Register Custom Dataset
5. Configure Training
6. Train Model
7. Evaluate Model
8. Export to OpenVINO
9. Download Model

---
## 🔧 STEP 1: Setup Environment

Install required packages and clone TorchReID repository.

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be slow.")
    print("Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install TorchReID
!pip install -q gdown
!git clone https://github.com/KaiyangZhou/deep-person-reid.git
%cd deep-person-reid
!pip install -q -r requirements.txt
!python setup.py develop -q
print("\n✅ TorchReID installed successfully!")

---
## 📁 STEP 2: Upload & Extract Dataset

Upload your `vehicle_reid_by_class.zip` file from Google Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("\n✅ Google Drive mounted!")

In [ ]:
# Configuration - UPDATE THIS PATH!
DATASET_ZIP_PATH = "/content/drive/MyDrive/vehicle_reid_by_class.zip"  # <-- UPDATE THIS

import os
import zipfile
import shutil

# Check if zip exists
if not os.path.exists(DATASET_ZIP_PATH):
    print(f"❌ ERROR: Dataset not found at: {DATASET_ZIP_PATH}")
    print("\nPlease upload your vehicle_reid_by_class.zip to Google Drive and update the path above.")
else:
    print(f"✅ Found dataset: {DATASET_ZIP_PATH}")
    print(f"   Size: {os.path.getsize(DATASET_ZIP_PATH) / 1e6:.1f} MB")
    
    # Extract dataset
    print("\nExtracting dataset...")
    DATASET_DIR = "/content/vehicle_reid_by_class"
    
    if os.path.exists(DATASET_DIR):
        shutil.rmtree(DATASET_DIR)
    
    with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    
    print(f"\n✅ Dataset extracted to: {DATASET_DIR}")

---
## 📊 STEP 3: Analyze Dataset

Check dataset structure and class distribution.

In [ ]:
import os
import glob
import re
from collections import defaultdict
import matplotlib.pyplot as plt

DATASET_DIR = "/content/vehicle_reid_by_class"

# Check directory structure
print("Dataset Structure:")
print("=" * 50)

for subdir in ['bounding_box_train', 'query', 'bounding_box_test']:
    path = os.path.join(DATASET_DIR, subdir)
    if os.path.exists(path):
        count = len(glob.glob(os.path.join(path, '*.jpg')))
        print(f"  {subdir}/: {count} images")
    else:
        print(f"  {subdir}/: NOT FOUND!")

print()

In [ ]:
# Analyze class distribution
train_dir = os.path.join(DATASET_DIR, 'bounding_box_train')
images = glob.glob(os.path.join(train_dir, '*.jpg'))

# Parse filenames to get class info
# Format: 000001_CAR_c1_s1.jpg
pattern = re.compile(r'(\d+)_([A-Z]+)_c(\d+)_s(\d+)')

class_counts = defaultdict(int)
vehicle_ids = set()

for img_path in images:
    fname = os.path.basename(img_path)
    match = pattern.search(fname)
    if match:
        vid = int(match.group(1))
        vclass = match.group(2)
        vehicle_ids.add(vid)
        class_counts[vclass] += 1

print("Class Distribution (Training Set):")
print("=" * 50)
for cls, count in sorted(class_counts.items()):
    print(f"  {cls}: {count} images")
print(f"\nTotal unique vehicles: {len(vehicle_ids)}")
print(f"Total training images: {len(images)}")

In [ ]:
# Visualize class distribution
import matplotlib.pyplot as plt

classes = list(class_counts.keys())
counts = [class_counts[c] for c in classes]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1.bar(classes, counts, color=colors)
ax1.set_xlabel('Vehicle Class')
ax1.set_ylabel('Number of Images')
ax1.set_title('Class Distribution')
for i, v in enumerate(counts):
    ax1.text(i, v + 50, str(v), ha='center', fontsize=10)

# Pie chart
ax2.pie(counts, labels=classes, autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Class Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# Show sample images from each class
import cv2
from PIL import Image
import random

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Images by Class', fontsize=14)

for idx, cls in enumerate(sorted(class_counts.keys())):
    # Get images for this class
    cls_images = [f for f in images if f'_{cls}_' in f]
    
    # Show 2 random samples
    for row in range(2):
        if cls_images:
            sample = random.choice(cls_images)
            img = Image.open(sample)
            axes[row, idx].imshow(img)
            axes[row, idx].set_title(f'{cls}', fontsize=10)
            axes[row, idx].axis('off')

plt.tight_layout()
plt.show()

---
## 📝 STEP 4: Register Custom Dataset

Create and register the custom vehicle dataset with TorchReID.

In [ ]:
import torchreid
from torchreid.data import ImageDataset
import os
import glob
import re

class CustomVehicleDataset(ImageDataset):
    """Custom Vehicle ReID Dataset
    
    Filename format: {VehicleID:06d}_{CLASS}_c{CameraID}_s{Scene}.jpg
    Example: 000001_CAR_c1_s1.jpg
    """
    dataset_dir = 'vehicle_reid_by_class'
    
    def __init__(self, root='/content', **kwargs):
        self.root = root
        self.dataset_dir = os.path.join(root, self.dataset_dir)
        
        if not os.path.exists(self.dataset_dir):
            raise RuntimeError(f"Dataset not found: {self.dataset_dir}")
        
        train_dir = os.path.join(self.dataset_dir, 'bounding_box_train')
        query_dir = os.path.join(self.dataset_dir, 'query')
        gallery_dir = os.path.join(self.dataset_dir, 'bounding_box_test')
        
        train = self.process_dir(train_dir)
        query = self.process_dir(query_dir)
        gallery = self.process_dir(gallery_dir)
        
        print(f"Dataset loaded:")
        print(f"  Train:   {len(train)} images")
        print(f"  Query:   {len(query)} images")
        print(f"  Gallery: {len(gallery)} images")
        
        super(CustomVehicleDataset, self).__init__(train, query, gallery, **kwargs)
    
    def process_dir(self, dir_path):
        """Process directory and extract (path, pid, camid) tuples"""
        img_paths = glob.glob(os.path.join(dir_path, '*.jpg'))
        
        # Pattern: 000001_CAR_c1_s1.jpg
        pattern = re.compile(r'(\d+)_[A-Z]+_c(\d+)_s\d+')
        
        data = []
        for img_path in img_paths:
            fname = os.path.basename(img_path)
            match = pattern.search(fname)
            if match:
                pid = int(match.group(1))   # Vehicle ID
                camid = int(match.group(2)) # Camera ID (1=entry, 2=exit)
                data.append((img_path, pid, camid))
        
        return data

# Register dataset
torchreid.data.register_image_dataset('custom_vehicle', CustomVehicleDataset)
print("\n✅ Custom dataset registered!")

---
## ⚙️ STEP 5: Configure Training

Set up training parameters and data loaders.

In [ ]:
# Training Configuration
CONFIG = {
    # Model
    'model_name': 'osnet_ain_x1_0',  # Best accuracy/speed tradeoff
    
    # Input size (match your images)
    'height': 384,
    'width': 384,
    
    # Training
    'max_epoch': 60,
    'batch_size': 32,
    'learning_rate': 0.0015,
    
    # Triplet loss
    'margin': 0.3,
    'weight_t': 1.0,  # Triplet loss weight
    'weight_x': 1.0,  # Softmax loss weight
    
    # Evaluation
    'eval_freq': 10,  # Evaluate every N epochs
    
    # Save
    'save_dir': 'log/vehicle_reid'
}

print("Training Configuration:")
print("=" * 50)
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Create data manager
print("Creating data manager...")

datamanager = torchreid.data.ImageDataManager(
    root='/content',
    sources='custom_vehicle',
    height=CONFIG['height'],
    width=CONFIG['width'],
    batch_size_train=CONFIG['batch_size'],
    batch_size_test=CONFIG['batch_size'] * 2,
    transforms=['random_flip', 'random_crop', 'random_erasing'],
    num_instances=4,
    train_sampler='RandomIdentitySampler'
)

print(f"\n✅ Data manager created!")
print(f"  Number of training identities (vehicles): {datamanager.num_train_pids}")
print(f"  Number of training cameras: {datamanager.num_train_cams}")

---
## 🏋️ STEP 6: Train Model

Build and train the OSNet-AIN model.

In [ ]:
# Build model
print(f"Building model: {CONFIG['model_name']}")

model = torchreid.models.build_model(
    name=CONFIG['model_name'],
    num_classes=datamanager.num_train_pids,
    loss='triplet',
    pretrained=True
)

# Move to GPU
if torch.cuda.is_available():
    model = model.cuda()

print(f"\n✅ Model built!")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Build optimizer and scheduler
optimizer = torchreid.optim.build_optimizer(
    model,
    optim='adam',
    lr=CONFIG['learning_rate']
)

scheduler = torchreid.optim.build_lr_scheduler(
    optimizer,
    lr_scheduler='single_step',
    stepsize=30
)

print("✅ Optimizer and scheduler created!")

In [ ]:
# Build training engine
engine = torchreid.engine.ImageTripletEngine(
    datamanager,
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    margin=CONFIG['margin'],
    weight_t=CONFIG['weight_t'],
    weight_x=CONFIG['weight_x']
)

print("✅ Training engine created!")

In [ ]:
# START TRAINING
print("=" * 70)
print(f"🏋️ STARTING TRAINING")
print(f"   Epochs: {CONFIG['max_epoch']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Learning rate: {CONFIG['learning_rate']}")
print("=" * 70)

engine.run(
    save_dir=CONFIG['save_dir'],
    max_epoch=CONFIG['max_epoch'],
    eval_freq=CONFIG['eval_freq'],
    print_freq=50,
    test_only=False,
    visrank=False
)

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE!")
print("=" * 70)

---
## 📈 STEP 7: Evaluate Model

Check model performance metrics.

In [ ]:
# Find the best model
import os
import glob

model_files = glob.glob(os.path.join(CONFIG['save_dir'], '*.pth.tar'))

if model_files:
    # Get the latest model
    best_model = max(model_files, key=os.path.getctime)
    print(f"Best model: {best_model}")
    print(f"Size: {os.path.getsize(best_model) / 1e6:.1f} MB")
else:
    print("❌ No model found! Training may not have completed.")

In [ ]:
# Load and evaluate the best model
print("Loading best model for evaluation...")

torchreid.utils.load_pretrained_weights(model, best_model)
model.eval()

# Run evaluation
print("\nRunning evaluation...")
print("=" * 70)

engine.run(
    save_dir=CONFIG['save_dir'],
    max_epoch=0,
    eval_freq=1,
    test_only=True
)

print("=" * 70)
print("\n✅ Evaluation complete!")
print("")
print("Key metrics:")
print("  - mAP: Mean Average Precision (higher = better)")
print("  - Rank-1: Top-1 accuracy (higher = better)")
print("  - Rank-5: Top-5 accuracy (higher = better)")

In [ ]:
# Visualize training curves (if log exists)
import matplotlib.pyplot as plt
import json

log_file = os.path.join(CONFIG['save_dir'], 'log.txt')

if os.path.exists(log_file):
    epochs = []
    losses = []
    ranks = []
    
    with open(log_file, 'r') as f:
        for line in f:
            if 'Epoch' in line and 'loss' in line.lower():
                # Parse epoch and loss from log
                pass  # Log format may vary
    
    print("Training log found. Check log/vehicle_reid/ for detailed metrics.")
else:
    print("Training log not found.")

---
## 🔄 STEP 8: Export to OpenVINO

Convert the trained model to OpenVINO format for fast CPU inference.

In [ ]:
# Install OpenVINO
!pip install -q openvino openvino-dev
print("✅ OpenVINO installed!")

In [ ]:
# Export to ONNX first
import torch.nn as nn

class FeatureExtractor(nn.Module):
    """Wrapper to extract features only"""
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        return self.model(x)

# Load model
torchreid.utils.load_pretrained_weights(model, best_model)
model.eval()
model.cpu()

# Wrap model
feature_extractor = FeatureExtractor(model)
feature_extractor.eval()

# Export to ONNX
onnx_path = os.path.join(CONFIG['save_dir'], 'vehicle_reid.onnx')
dummy_input = torch.randn(1, 3, CONFIG['height'], CONFIG['width'])

print("Exporting to ONNX...")
torch.onnx.export(
    feature_extractor,
    dummy_input,
    onnx_path,
    opset_version=11,
    input_names=['input'],
    output_names=['embeddings'],
    dynamic_axes={'input': {0: 'batch'}, 'embeddings': {0: 'batch'}}
)

print(f"✅ ONNX saved: {onnx_path}")
print(f"   Size: {os.path.getsize(onnx_path) / 1e6:.1f} MB")

In [ ]:
# Convert to OpenVINO
from openvino.tools import mo
from openvino.runtime import serialize

print("Converting to OpenVINO...")

ov_model = mo.convert_model(onnx_path, compress_to_fp16=True)

xml_path = os.path.join(CONFIG['save_dir'], 'vehicle_reid_FP16.xml')
bin_path = os.path.join(CONFIG['save_dir'], 'vehicle_reid_FP16.bin')

serialize(ov_model, xml_path)

print(f"\n✅ OpenVINO model saved!")
print(f"   XML: {xml_path}")
print(f"   BIN: {bin_path}")
print(f"   Size: {os.path.getsize(xml_path) / 1e6 + os.path.getsize(bin_path) / 1e6:.1f} MB")

In [ ]:
# Verify OpenVINO model
from openvino.runtime import Core
import numpy as np

print("Verifying OpenVINO model...")

ie = Core()
compiled = ie.compile_model(xml_path, "CPU")

# Test inference
test_input = np.random.randn(1, 3, CONFIG['height'], CONFIG['width']).astype(np.float32)
result = compiled([test_input])

print(f"\n✅ OpenVINO verification successful!")
print(f"   Input shape: {test_input.shape}")
print(f"   Output shape: {result[0].shape}")
print(f"   Embedding dimension: {result[0].shape[1]}")

---
## 📥 STEP 9: Download Model

Save all model files to Google Drive and download.

In [ ]:
# Copy models to Google Drive
import shutil

drive_output = "/content/drive/MyDrive/vehicle_reid_trained"
os.makedirs(drive_output, exist_ok=True)

# Copy files
files_to_copy = [
    best_model,
    onnx_path,
    xml_path,
    bin_path
]

print("Copying models to Google Drive...")
for src in files_to_copy:
    if os.path.exists(src):
        dst = os.path.join(drive_output, os.path.basename(src))
        shutil.copy(src, dst)
        print(f"  ✓ {os.path.basename(src)}")

print(f"\n✅ Models saved to: {drive_output}")

In [ ]:
# Create a zip file for easy download
zip_output = "/content/vehicle_reid_models.zip"

!cd {CONFIG['save_dir']} && zip -r {zip_output} vehicle_reid_FP16.xml vehicle_reid_FP16.bin vehicle_reid.onnx model.pth.tar

print(f"\n✅ Zip created: {zip_output}")

# Download link
from google.colab import files
print("\nDownloading...")
files.download(zip_output)

---
## 📋 Summary

### What you trained:
- **Model**: OSNet-AIN (Omni-Scale Network with Instance Normalization)
- **Input**: 384x384 vehicle images
- **Output**: 512-dimensional embedding vector
- **Classes**: CAR, LCV, TRUCK, BUS

### Files created:
- `model.pth.tar` - PyTorch model (for fine-tuning)
- `vehicle_reid.onnx` - ONNX model (portable)
- `vehicle_reid_FP16.xml/bin` - OpenVINO model (fast CPU inference)

### To use the model:
1. Copy `vehicle_reid_FP16.xml` and `vehicle_reid_FP16.bin` to your server
2. Update `config.ini`:
   ```ini
   [reid]
   openvino_model = models/reid/vehicle_reid_FP16.xml
   ```
3. Restart your service

### Performance tips:
- Larger batch sizes = faster training
- More epochs = better accuracy (up to a point)
- Add more data for better generalization